# Results Analysis & Visualization

Final analysis notebook for the halo-mass regression and explainability pipeline. It summarizes model performance, prediction behavior, XAI outputs, radial attribution metrics, error patterns, and paper-ready conclusions.

**Scientific framing:** this notebook analyzes regions influential for predicting halo-related physical properties. It does not claim direct dark matter detection.

## 1. Setup & Load Results

This section loads whichever artifacts are available. Missing files are reported clearly so the notebook can be run progressively while the pipeline is still being built.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid')
except Exception:
    sns = None

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = Path('outputs/regression_resnet18')
RESULTS_DIR = Path('results')
XAI_DIR = RESULTS_DIR / 'xai'
ANALYSIS_DIR = RESULTS_DIR / 'analysis'
FINAL_DIR = RESULTS_DIR / 'final'
FINAL_DIR.mkdir(parents=True, exist_ok=True)

ARTIFACTS = {
    'metrics_test': OUTPUT_DIR / 'metrics_test.json',
    'predictions_test': OUTPUT_DIR / 'predictions_test.csv',
    'training_history': OUTPUT_DIR / 'training_history.csv',
    'xai_summary': XAI_DIR / 'xai_summary.csv',
    'radial_importance': ANALYSIS_DIR / 'radial_importance.csv',
}

def exists(path):
    return Path(path).exists()

print('Artifact availability:')
for name, path in ARTIFACTS.items():
    print(f'  {name:18s}: {path} | exists={exists(path)}')

metrics = {}
if exists(ARTIFACTS['metrics_test']):
    metrics = json.loads(ARTIFACTS['metrics_test'].read_text())

predictions_df = pd.read_csv(ARTIFACTS['predictions_test']) if exists(ARTIFACTS['predictions_test']) else pd.DataFrame()
history_df = pd.read_csv(ARTIFACTS['training_history']) if exists(ARTIFACTS['training_history']) else pd.DataFrame()
xai_df = pd.read_csv(ARTIFACTS['xai_summary']) if exists(ARTIFACTS['xai_summary']) else pd.DataFrame()
radial_df = pd.read_csv(ARTIFACTS['radial_importance']) if exists(ARTIFACTS['radial_importance']) else pd.DataFrame()

print('\nLoaded tables:')
print(f'  predictions rows: {len(predictions_df)}')
print(f'  history rows:     {len(history_df)}')
print(f'  xai rows:         {len(xai_df)}')
print(f'  radial rows:      {len(radial_df)}')

## 2. Model Performance Metrics

In [ ]:
if metrics:
    metrics_df = pd.DataFrame([metrics])
    display(metrics_df)
else:
    print('metrics_test.json not found. Run training/evaluation first.')

if not history_df.empty and {'epoch', 'train_loss', 'val_loss'}.issubset(history_df.columns):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(history_df['epoch'], history_df['train_loss'], label='train_loss')
    ax.plot(history_df['epoch'], history_df['val_loss'], label='val_loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training and validation loss')
    ax.legend()
    fig.tight_layout()
    out_path = FINAL_DIR / 'training_loss_summary.png'
    fig.savefig(out_path, dpi=160)
    plt.show()
    print(f'Saved: {out_path}')
else:
    print('Training history not available or missing expected columns.')

## 3. Prediction Analysis

This section compares predicted and true log10 halo mass values. Good performance should show points close to the one-to-one line and residuals centered near zero.

In [ ]:
if not predictions_df.empty and {'y_true', 'y_pred'}.issubset(predictions_df.columns):
    predictions_df = predictions_df.copy()
    predictions_df['residual'] = predictions_df['y_pred'] - predictions_df['y_true']
    predictions_df['absolute_error'] = predictions_df['residual'].abs()

    display(predictions_df.head())

    y_true = predictions_df['y_true']
    y_pred = predictions_df['y_pred']
    low = min(y_true.min(), y_pred.min())
    high = max(y_true.max(), y_pred.max())

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].scatter(y_true, y_pred, alpha=0.75, s=28)
    axes[0].plot([low, high], [low, high], color='black', linewidth=1.5)
    axes[0].set_xlabel('True log10 halo mass')
    axes[0].set_ylabel('Predicted log10 halo mass')
    axes[0].set_title('Predicted vs true')

    axes[1].hist(predictions_df['residual'], bins=20, alpha=0.85)
    axes[1].axvline(0, color='black', linewidth=1.5)
    axes[1].set_xlabel('Prediction residual')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Residual distribution')

    fig.tight_layout()
    out_path = FINAL_DIR / 'prediction_analysis.png'
    fig.savefig(out_path, dpi=160)
    plt.show()
    print(f'Saved: {out_path}')
else:
    print('Prediction CSV not found. Run src.training.evaluate_regression first.')

## 4. XAI Attribution Analysis

Attribution maps are interpreted as regions influential for the model prediction, not as direct dark matter maps.

In [ ]:
if not xai_df.empty:
    display(xai_df.head())
    print('XAI samples:', len(xai_df))
    if 'note' in xai_df.columns:
        print('Scientific note:', xai_df['note'].dropna().iloc[0] if xai_df['note'].notna().any() else 'No note stored')
else:
    print('xai_summary.csv not found. Generate XAI maps first with notebooks/08_generate_xai_maps.ipynb.')

xai_pngs = sorted(XAI_DIR.glob('*.png')) if XAI_DIR.exists() else []
print(f'Available XAI PNG files: {len(xai_pngs)}')
for path in xai_pngs[:10]:
    print(' ', path)

## 5. Radial Importance Analysis

These metrics answer the central research question: in which image regions is the predictive information concentrated? The regions are geometric center/middle/outer masks and are not physical dark matter labels.

In [ ]:
region_cols = ['center_importance_percent', 'middle_importance_percent', 'outer_importance_percent']

if not radial_df.empty and set(region_cols).issubset(radial_df.columns):
    display(radial_df.head())
    radial_summary = radial_df.groupby('method')[region_cols].agg(['mean', 'std', 'median'])
    display(radial_summary)

    mean_df = radial_df.groupby('method')[region_cols].mean()
    plot_df = mean_df.rename(columns={
        'center_importance_percent': 'center',
        'middle_importance_percent': 'middle',
        'outer_importance_percent': 'outer',
    })

    ax = plot_df.T.plot(kind='bar', figsize=(8, 5))
    ax.set_ylabel('Mean attribution importance (%)')
    ax.set_xlabel('Radial region')
    ax.set_title('Mean attribution by radial region')
    ax.legend(title='Method')
    plt.tight_layout()
    out_path = FINAL_DIR / 'radial_importance_summary.png'
    plt.savefig(out_path, dpi=160)
    plt.show()
    print(f'Saved: {out_path}')
else:
    print('Radial importance CSV not found. Run notebooks/09_radial_importance_analysis.ipynb first.')

## 6. Error Analysis

This section identifies high-error samples and checks whether error changes with halo mass. These diagnostics help decide whether additional data filtering, augmentation, or architecture changes are needed.

In [ ]:
if not predictions_df.empty and {'y_true', 'y_pred'}.issubset(predictions_df.columns):
    predictions_df = predictions_df.copy()
    predictions_df['residual'] = predictions_df['y_pred'] - predictions_df['y_true']
    predictions_df['absolute_error'] = predictions_df['residual'].abs()

    worst = predictions_df.sort_values('absolute_error', ascending=False).head(10)
    print('Highest-error samples:')
    display(worst)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(predictions_df['y_true'], predictions_df['absolute_error'], alpha=0.75, s=28)
    ax.set_xlabel('True log10 halo mass')
    ax.set_ylabel('Absolute error')
    ax.set_title('Absolute error vs true halo mass')
    fig.tight_layout()
    out_path = FINAL_DIR / 'error_vs_halo_mass.png'
    fig.savefig(out_path, dpi=160)
    plt.show()
    print(f'Saved: {out_path}')
else:
    print('Prediction data unavailable for error analysis.')

## 7. Final Report & Conclusions

In [ ]:
def fmt_metric(name, value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return f'{name}: unavailable'
    return f'{name}: {value:.4f}'

lines = []
lines.append('# Final Results Summary')
lines.append('')
lines.append('## Scientific Framing')
lines.append('The project studies whether galaxy images contain spatially localized information predictive of halo-related physical properties. Attribution maps are not direct dark matter detections.')
lines.append('')
lines.append('## Regression Performance')
if metrics:
    lines.append('- ' + fmt_metric('MAE', metrics.get('mae')))
    lines.append('- ' + fmt_metric('RMSE', metrics.get('rmse')))
    lines.append('- ' + fmt_metric('R2', metrics.get('r2')))
else:
    lines.append('- Metrics unavailable.')

lines.append('')
lines.append('## Prediction Diagnostics')
if not predictions_df.empty and 'absolute_error' in predictions_df.columns:
    lines.append(f'- Evaluated samples: {len(predictions_df)}')
    lines.append(f'- Mean absolute error from prediction table: {predictions_df["absolute_error"].mean():.4f}')
    lines.append(f'- Median absolute error: {predictions_df["absolute_error"].median():.4f}')
else:
    lines.append('- Prediction table unavailable.')

lines.append('')
lines.append('## XAI and Radial Interpretation')
if not radial_df.empty and set(region_cols).issubset(radial_df.columns):
    means = radial_df[region_cols].mean()
    dominant = means.idxmax().replace('_importance_percent', '')
    lines.append(f'- Mean center attribution: {means["center_importance_percent"]:.2f}%')
    lines.append(f'- Mean middle attribution: {means["middle_importance_percent"]:.2f}%')
    lines.append(f'- Mean outer attribution: {means["outer_importance_percent"]:.2f}%')
    lines.append(f'- Dominant radial region in available attribution maps: {dominant}')
else:
    lines.append('- Radial attribution metrics unavailable.')

lines.append('')
lines.append('## Conclusion')
lines.append('The completed pipeline supports image-to-halo-property regression, explainability map generation, and radial attribution analysis. The interpretation should remain statistical and attribution-based: highlighted regions indicate influence on the trained model prediction, not observed dark matter locations.')

report = '\n'.join(lines)
report_path = FINAL_DIR / 'final_results_summary.md'
report_path.write_text(report, encoding='utf-8')
print(report)
print(f'\nSaved final report: {report_path}')

## Expected output

The notebook saves final analysis figures and results/final/final_results_summary.md. It summarizes regression performance, prediction errors, XAI artifacts, radial importance, and scientific conclusions without claiming direct dark matter detection.